### Setting paths and imports

In [1]:
import sys
import os
import pandas as pd

module_path = os.path.abspath(os.path.join('..'))
sys.path.append(module_path)

os.chdir(module_path)

In [2]:
from src.utils.context import Context
from src.data_analysis.future.data_analyzer import DataAnalyzer as data_analyzer
from src.explainer.future.metaheuristic.aco.hypercube_aco import HypercubeACOExplainer
from src.evaluation.future.evaluator_manager_triplets import EvaluatorManager

config_f_name = 'metaheuristics/ACO/hypercube_aco_pipeline_asd.jsonc'

In [3]:
HypercubeACOExplainer._cache_log.clear()

### Evaluating the explainer

In [4]:
config_path = os.path.join(module_path, 'lab', 'config', config_f_name)
runno = 1
    
print(f"Generating context for: {config_path}")
context = Context.get_context(config_path)
context.run_number = runno

context.logger.info(f"Executing: {context.config_file} Run: {context.run_number}")
context.logger.info("Creating the evaluation manager....................................")

context.logger.info("Creating the evaluators......................................................")
eval_manager = EvaluatorManager(context)

context.logger.info(
    "Evaluating the explainers............................................................."
)
eval_manager.evaluate()

Generating context for: /home/jorge/Escuela/gretel3/GRETEL/lab/config/metaheuristics/ACO/hypercube_aco_pipeline_asd.jsonc
2026-06-19 22:33:16,050 | INFO | 122917 - Executing: /home/jorge/Escuela/gretel3/GRETEL/lab/config/metaheuristics/ACO/hypercube_aco_pipeline_asd.jsonc Run: 1
2026-06-19 22:33:16,063 | INFO | 122917 - Creating the evaluation manager....................................
2026-06-19 22:33:16,075 | INFO | 122917 - Creating the evaluators......................................................
2026-06-19 22:33:16,103 | INFO | 122917 - Loading: ASD-a90450198d54af7520c0a579af443e89
2026-06-19 22:33:16,159 | INFO | 122917 - Created: ASD-a90450198d54af7520c0a579af443e89
2026-06-19 22:33:16,176 | INFO | 122917 - Loading: ASDOracle-12b3e9daa577d8245fb63e12b30ea64d
2026-06-19 22:33:16,189 | INFO | 122917 - Created: ASDOracle-12b3e9daa577d8245fb63e12b30ea64d
2026-06-19 22:33:16,201 | INFO | 122917 - Created: HypercubeACOExplainer-54736dc4e25c8979b04807c6a007b530
2026-06-19 22:33:16,

In [5]:
exp_inst = []
for exp in eval_manager.evaluators[0]._explanations:
    exp.input_instance._dataset = None
    exp.counterfactual_instances[0]._dataset = None
    
exp_inst = [(exp.input_instance, exp.counterfactual_instances[0]) for exp in eval_manager.evaluators[0]._explanations]

### Aggregating the stats

In [6]:
results_path = os.path.join(module_path, 'lab', 'output', 'results')
stats_file_path = os.path.join(module_path, 'lab', 'stats', 'results.csv')
res = data_analyzer.create_aggregated_dataframe(results_path)
res.to_csv(stats_file_path)
res

,scope,dataset,oracle,explainer,Runtime,Runtime-std,GraphEditDistance,GraphEditDistance-std,Correctness,Correctness-std,OracleCalls,OracleCalls-std,OracleAccuracy,OracleAccuracy-std,Sparsity,Sparsity-std,Fidelity,Fidelity-std
0,hypercube_aco_asd_experiments,ASD-a90450198d54af7520c0a579af443e89,ASDOracle-12b3e9daa577d8245fb63e12b30ea64d,HypercubeACOExplainer,15.075520,1.472507,9.162466,0.746625,1.000000,0.000000,1035.706571,38.657066,0.79523,0.022952,0.020738,0.001764,0.590459,0.045905
1,hypercube_aco_asd_experiments,ASD-a90450198d54af7520c0a579af443e89,ASDOracle-12b3e9daa577d8245fb63e12b30ea64d,IRandExplainer,1.317758,0.062582,35.411585,11.661585,0.384788,0.021152,286.040054,7.494599,0.79523,0.022952,0.079866,0.026311,0.024752,0.024752
2,hypercube_aco_asd_experiments,ASD-a90450198d54af7520c0a579af443e89,ASDOracle-12b3e9daa577d8245fb63e12b30ea64d,ObliviousBidirectionalSearchExplainer,0.575147,0.004643,10.895140,1.013951,1.000000,0.000000,421.093159,41.815932,0.79523,0.022952,0.024653,0.002377,0.590459,0.045905


In [7]:
import pandas as pd

df = pd.DataFrame(
    HypercubeACOExplainer._cache_log,
    columns=["cache_hits", "oracle_calls"]
)

total = df["cache_hits"] + df["oracle_calls"]
df["hit_rate"] = df["cache_hits"].div(total).fillna(0)

global_hit_rate = df["cache_hits"].sum() / total.sum() if total.sum() else 0

print(f"Instancias: {len(df)}")
print(f"Cache hits promedio: {df['cache_hits'].mean():.3f}")
print(f"Hit rate promedio: {df['hit_rate'].mean():.3f}")
print(f"Hit rate global: {global_hit_rate:.3f}")

Instancias: 101
Cache hits promedio: 85.475
Hit rate promedio: 0.113
Hit rate global: 0.079


In [8]:
# beep
from IPython.display import Audio, display
import numpy as np

t = np.linspace(0, 0.3, 44100, False)
tone = np.sin(880 * 2 * np.pi * t)
display(Audio(tone, rate=44100, autoplay=True))
